# CogniFusion — Colab Notebook

**Real-time cognitive state monitoring** that fuses EEG (Muse-style 4-channel)
and eye-tracking signals to classify **low / medium / high** cognitive load.

| | |
|---|---|
| Repo | [PoornaSri26/CogniFusion](https://github.com/PoornaSri26/CogniFusion) |
| Runtime | CPU works — pick **GPU** (Runtime → Change runtime type) to train faster |
| Data | Synthetic 30-subject dataset generated in-notebook; no hardware needed |

## What this notebook does

1. **Setup** — clone repo, install deps
2. **Data** — generate 360 synchronized EEG+eye epochs (30 subjects × 3 classes)
3. **Features** — extract 168 fused features per epoch (137 EEG + 31 eye)
4. **Explore** — signal plots, feature correlations, class separation
5. **Train** — `AttentionFusionNet`: vector-gated cross-modal attention fusion,
   focal loss, mixup augmentation, per-subject median/MAD normalization,
   temperature scaling, EMA smoothing, optional LightGBM stacking
6. **Evaluate** — *every model in the ensemble* scored individually, then the
   full ensemble, on subjects the model has never seen

## Reading the results

- **Subject-wise holdout** = the honest metric. Test subjects never appear in
  training, so it measures generalization to new people — not memorization.
- Synthetic data is clean and class-separable, so expect high scores here;
  real Muse+webcam data is harder (see the repo README for real-data results).

## 1 · Setup — clone repo & install deps

In [ ]:
# Clone the repository
!git clone -q https://github.com/PoornaSri26/CogniFusion.git
%cd CogniFusion

# Core ML deps + lightgbm for the stacked ensemble.
# torch is preinstalled on Colab — don't reinstall it.
!pip install -q numpy pandas scipy scikit-learn pyarrow lightgbm

import sys, torch
sys.path.insert(0, "src")
sys.path.insert(0, "data")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2 · Generate the synthetic dataset

`data/simulate_data.py` synthesizes EEG (4 channels @ 256 Hz, 4 s epochs) and
eye-tracking (60 Hz: pupil, gaze, fixation, blinks) for 30 subjects × 4 epochs ×
3 classes = **360 epochs**.

Each subject gets unique biological parameters — individual alpha frequency
(9–12 Hz), noise floor, electrode offsets, pupil baseline, blink rate — so the
model must learn *subject-invariant* load features rather than memorizing noise.

In [ ]:
from pathlib import Path
from simulate_data import generate_dataset

if Path("data/raw/manifest.csv").exists():
    print("data/raw/ already populated — skipping generation")
    import pandas as pd
    manifest = pd.read_csv("data/raw/manifest.csv")
else:
    manifest = generate_dataset("data/raw")
manifest.head()


## 3 · Build the fused feature table

`src/build_dataset.py` runs feature extraction on every epoch and joins EEG +
eye features into one parquet table:

- **EEG** (`eeg_features.py`): band powers (delta/theta/alpha/beta/gamma),
  differential entropy, frontal/parietal ratios, spectral entropy, Hjorth
  params, coherence & phase-lag connectivity, fractal dimension
- **Eye** (`eye_features.py`): pupil diameter stats & variability, fixation
  ratio, blink rate, gaze dispersion/speed/acceleration, microsaccades

> This is the slowest step (~10 min on CPU) — extraction is per-epoch.
>
> **Already have data?** Drop your own `manifest.csv` + raw epochs into
> `data/raw/` (or a ready `features.parquet` into `data/processed/`) and this
> cell — like data generation above — skips straight to using it.

In [ ]:
from pathlib import Path
import pandas as pd

FEAT = Path("data/processed/features.parquet")
if FEAT.exists():
    print("features.parquet already exists — skipping feature extraction")
    df = pd.read_parquet(FEAT)
else:
    from build_dataset import build
    df = build("data/raw/manifest.csv", str(FEAT))
df.shape


## 3b · Preprocessing & data-quality checks

Before modeling: missing values, outliers (IQR fences), and per-class
feature drift. `build_dataset_real.py` already median-imputes NaNs; this cell
shows you *what* needed cleaning — run it on `features_real.parquet` after
section 12 to audit the real data.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

FEAT = "data/processed/features_real.parquet" if Path("data/processed/features_real.parquet").exists() else "data/processed/features.parquet"
df = pd.read_parquet(FEAT)
feat_cols = [c for c in df.columns if c.startswith(("eeg__","eye__"))]
print(f"Table: {FEAT}  |  {df.shape[0]} epochs x {len(feat_cols)} features")

# 1) Missing values
nan_per_col = df[feat_cols].isna().sum()
print("Columns with NaNs:", int((nan_per_col>0).sum()), "| total NaN cells:", int(nan_per_col.sum()))
if (nan_per_col>0).any():
    display(nan_per_col[nan_per_col>0].sort_values(ascending=False).head(10).to_frame("NaN count"))

# 2) Outliers via IQR fences
q1, q3 = df[feat_cols].quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = (df[feat_cols] < (q1 - 1.5*iqr)) | (df[feat_cols] > (q3 + 1.5*iqr))
outlier_rate = outlier_mask.mean().sort_values(ascending=False)
print("Top-10 columns by outlier rate (>1.5xIQR):")
display((outlier_rate.head(10)*100).round(2).to_frame("% rows outlying"))
print(f"Median outlier rate across features: {outlier_rate.median()*100:.2f}%")

# 3) Class-conditional drift on top features
load = df["label"].map({"low":0,"medium":1,"high":2})
corr = df[feat_cols].corrwith(load).abs().sort_values(ascending=False)
top5 = corr.head(5).index
fig, axes = plt.subplots(1, len(top5), figsize=(3*len(top5), 3), sharey=False)
for ax, col in zip(axes, top5):
    for lab in ["low","medium","high"]:
        v = df.loc[df["label"]==lab, col].dropna()
        ax.hist(v, bins=25, alpha=0.55, label=lab, density=True)
    ax.set_title(col.replace("eeg__","").replace("eye__",""), fontsize=8)
    ax.tick_params(labelsize=7)
axes[0].legend(fontsize=7)
plt.suptitle("Class-conditional distributions — top 5 load-correlated features", fontsize=10)
plt.tight_layout(); plt.show()

# 4) Optional: clip extreme outliers (uncomment to apply)
# df[feat_cols] = df[feat_cols].clip(lower=q1-3*iqr, upper=q3+3*iqr)

## 4 · Dataset overview

In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/features.parquet")
print("Label distribution:")
display(df["label"].value_counts().to_frame("count"))

feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
print(f"\n{len(feat_cols)} fused features | missing values: {int(df[feat_cols].isna().sum().sum())}")

load = df["label"].map({"low": 0, "medium": 1, "high": 2})
top = df[feat_cols].corrwith(load).abs().sort_values(ascending=False).head(10)
print("\nTop features by |correlation| with cognitive load:")
display(top.to_frame("|r|"))

## 5 · Signal explorer — what does load look like in the raw signals?

One epoch per class: EEG time series + power spectral density (1–45 Hz),
then pupil diameter and gaze position. Expect alpha to drop and theta/beta to
rise as load increases.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch

manifest = pd.read_csv("data/raw/manifest.csv")
CHANNELS = ["AF7", "AF8", "TP9", "TP10"]

fig, axes = plt.subplots(3, 3, figsize=(17, 9))
for col, label in enumerate(["low", "medium", "high"]):
    row = manifest[manifest["label"] == label].iloc[0]
    eeg = np.load(row["eeg_path"])                      # (4, 1024)
    t = np.arange(eeg.shape[1]) / 256.0

    for ch in range(4):
        axes[0, col].plot(t, eeg[ch], lw=0.5, label=CHANNELS[ch])
    axes[0, col].set_title(f"EEG time series — {label}")
    axes[0, col].legend(fontsize=6, ncol=2)

    for ch in range(4):
        f, p = welch(eeg[ch], fs=256, nperseg=256)
        axes[1, col].semilogy(f, p, lw=0.8, label=CHANNELS[ch])
    axes[1, col].set_xlim(1, 45); axes[1, col].set_xlabel("Hz")
    axes[1, col].set_title("EEG power spectral density")
    axes[1, col].legend(fontsize=6, ncol=2)

    eye = pd.read_csv(row["eye_path"])
    axes[2, col].plot(eye["t"], eye["pupil_mm"], lw=0.6, color="steelblue", label="pupil mm")
    ax2 = axes[2, col].twinx()
    ax2.plot(eye["t"], eye["gaze_x"], lw=0.4, alpha=0.6, color="darkorange", label="gaze x")
    axes[2, col].set_title(f"Eye — pupil & gaze ({label})")
    axes[2, col].set_xlabel("s")
plt.tight_layout(); plt.show()

## 6 · Feature landscape — which features carry the load signal?

Left: top features ranked by correlation with load. Right: class means for the
top 6 — watch theta/beta rise and alpha fall as load increases.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_parquet("data/processed/features.parquet")
feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
load = df["label"].map({"low": 0, "medium": 1, "high": 2})

corr = df[feat_cols].corrwith(load)
top12 = corr.abs().sort_values(ascending=False).head(12).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr.loc[top12].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Top 12 features — correlation with cognitive load")
axes[0].set_xlabel("Pearson r"); axes[0].axvline(0, color="k", lw=0.6)

top6 = corr.abs().sort_values(ascending=False).head(6).index
df.groupby("label")[list(top6)].mean().T.plot(kind="bar", ax=axes[1],
                                             figsize=(12, 5), colormap="viridis")
axes[1].set_title("Class means — top 6 features"); axes[1].set_ylabel("mean value")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 7 · Train the model

`src/train.py` trains `AttentionFusionNet` with:

- **Vector-gated cross-modal fusion** — per-dimension trust weighting between
  EEG and eye embeddings (the model learns *when* to trust each modality)
- **Focal loss** (γ=2) targeting the ambiguous medium class
- **Mixup augmentation**, label smoothing, modality dropout
- **Per-subject median/MAD normalization** — removes subject baselines so the
  model generalizes to unseen people
- **Temperature scaling + EMA smoothing** — calibrated, jitter-free predictions
- **LightGBM stacking** (`--use-gbm`) — 60% NN + 40% GBM probability blend

Single subject-wise holdout by default. Add `--cv 5` for 5-fold CV (~5× slower).

In [ ]:
# 5 folds x 200 epochs - use a GPU runtime for reasonable speed.
# On CPU, drop to --cv 0 (single holdout) or --epochs 100.
!python src/train.py --features data/processed/features.parquet --epochs 200 --cv 5 --use-gbm --out models/fusion_model


In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print(json.dumps(ckpt["cv_summary"], indent=2))
print("\nPer-fold validation scores stored in checkpoint:")
for i, rec in enumerate(ckpt["folds"]):
    print(f"  fold {i+1}: acc={rec['val_accuracy']:.3f}  "
          f"macro-F1={rec['val_macro_f1']:.3f}  "
          f"lgb={'yes' if rec.get('lgb_model') else 'no'}")

## 8 · Training curves

Loss & validation macro-F1 every 10 epochs, recorded in the checkpoint.

In [ ]:
import torch
import matplotlib.pyplot as plt

ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
for i, rec in enumerate(ckpt["folds"]):
    h = rec.get("history") or []
    if not h:
        print(f"Fold {i+1}: no history recorded — retrain to enable curves")
        continue
    ep   = [d["epoch"] for d in h]
    loss = [d["loss"]   for d in h]
    f1   = [d["val_f1"] for d in h]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, loss, marker="o", ms=3, color="steelblue")
    ax[0].set_title(f"Fold {i+1} — training loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(ep, f1, marker="o", ms=3, color="seagreen")
    ax[1].axhline(0.9, ls="--", c="r", alpha=0.4, label="target 0.9")
    ax[1].set_title("validation macro-F1"); ax[1].set_xlabel("epoch")
    ax[1].legend(); plt.tight_layout(); plt.show()

## 9 · Every model in the ensemble — scored individually, then pooled

With `--cv 5` the checkpoint holds **5 folds × (NN + LightGBM)** = 10 models.
This cell reproduces each fold's *own* held-out subject split (no leakage —
a fold is only scored on subjects it never trained on), then pools all
predictions for the headline CV score.

Per fold it reports **NN-only vs NN+GBM-stacked** accuracy, so you can see
exactly what the LightGBM blend buys you.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

from model import AttentionFusionNet
from train import (subject_wise_split, subject_kfold, adaptive_subject_norm,
                   coerce_numeric, make_norm_key, TemperatureScaler)

def load_ensemble(path="models/fusion_model.pt"):
    # Load the self-contained ensemble checkpoint (same logic as
    # realtime_pipeline.load_model, without hardware deps like brainflow).
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    folds = []
    for rec in ckpt["folds"]:
        m = AttentionFusionNet(
            eeg_dim=len(ckpt["eeg_cols"]),
            eye_dim=len(ckpt["eye_cols"]),
            embed_dim=rec.get("embed_dim", ckpt.get("embed_dim", 48)),
            n_classes=len(ckpt["label_classes"]),
            gate_type=ckpt.get("gate_type", "vector"),
            seq_len=rec.get("seq_len", 1),
        )
        m.load_state_dict(rec["model_state"])
        m.eval()
        ts = TemperatureScaler()
        ts.load_state_dict(rec["temp_scaler"])
        ts.eval()
        folds.append({"model": m, "scaler_eeg": rec["scaler_eeg"],
                      "scaler_eye": rec["scaler_eye"], "temp": ts,
                      "lgb": rec.get("lgb_model")})
    return {"folds": folds, "eeg_cols": ckpt["eeg_cols"],
            "eye_cols": ckpt["eye_cols"],
            "label_classes": list(ckpt["label_classes"])}

bundle = load_ensemble("models/fusion_model.pt")
eeg_cols, eye_cols, classes = bundle["eeg_cols"], bundle["eye_cols"], bundle["label_classes"]

df = pd.read_parquet("data/processed/features.parquet")
df["norm_key"] = make_norm_key(df)

# --- Rebuild the SAME splits training used (k = number of folds) ---
k = len(bundle["folds"])
if k > 1:
    splits = list(subject_kfold(df, k=k, seed=42))
    print(f"{k}-fold subject-wise CV — each fold scored on its own unseen subjects")
else:
    splits = [subject_wise_split(df, test_frac=0.25, seed=42)]
    print("Single subject-wise holdout split")

def ema(raw):
    out = np.zeros_like(raw)
    out[0] = raw[0]
    for t in range(1, len(raw)):
        out[t] = 0.65 * raw[t] + 0.35 * out[t - 1]
    return out

rows, pooled = [], {"y": [], "pred": [], "conf": [], "subject": [], "label": [],
                    "epoch_id": [], "probs": []}
with torch.no_grad():
    for i, fold in enumerate(bundle["folds"]):
        _, te = splits[i]
        te = te.copy()
        te["norm_key"] = make_norm_key(te)
        y_te = te["label"].to_numpy()

        eeg_n, _ = adaptive_subject_norm(coerce_numeric(te, eeg_cols), te["norm_key"])
        eye_n, _ = adaptive_subject_norm(coerce_numeric(te, eye_cols), te["norm_key"])
        xe = np.nan_to_num(fold["scaler_eeg"].transform(eeg_n))
        ye = np.nan_to_num(fold["scaler_eye"].transform(eye_n))

        # NN-only probabilities (temperature-calibrated)
        logits = fold["model"](torch.tensor(xe, dtype=torch.float32),
                               torch.tensor(ye, dtype=torch.float32))
        logits = fold["temp"](logits)
        nn_p = torch.softmax(logits, dim=1).numpy()

        # Stacked: 60% NN + 40% LightGBM (same blend as realtime_pipeline)
        stack_p = nn_p
        if fold["lgb"] is not None:
            lgb_p = fold["lgb"].predict_proba(np.concatenate([xe, ye], axis=1))
            stack_p = 0.60 * nn_p + 0.40 * lgb_p

        for tag, p in (("nn", nn_p), ("nn+gbm", stack_p)):
            pred_i = np.array(classes)[p.argmax(1)]
            if tag == "nn+gbm":
                rows.append({"fold": i + 1,
                             "nn_acc": round(accuracy_score(y_te, np.array(classes)[nn_p.argmax(1)]), 4),
                             "nn_f1":  round(f1_score(y_te, np.array(classes)[nn_p.argmax(1)], average="macro"), 4),
                             "stack_acc": round(accuracy_score(y_te, pred_i), 4),
                             "stack_f1":  round(f1_score(y_te, pred_i, average="macro"), 4)})

        # pooled predictions use the stacked probs + EMA
        pe = ema(stack_p)
        pooled["y"] += list(y_te)
        pooled["pred"] += list(np.array(classes)[pe.argmax(1)])
        pooled["conf"] += list(pe.max(1))
        pooled["subject"] += list(te["subject"])
        pooled["label"] += list(te["label"])
        pooled["epoch_id"] += list(te["epoch_id"])
        pooled["probs"] += list(pe)

print("\n=== Per-fold scores (each fold on its own held-out subjects) ===")
display(pd.DataFrame(rows))

y_all = np.array(pooled["y"]); p_all = np.array(pooled["pred"]); c_all = np.array(pooled["conf"])
print("\n=== Pooled CV score (every subject predicted once, by its held-out fold) ===")
print(f"accuracy    : {accuracy_score(y_all, p_all):.1%}")
print(f"macro-F1    : {f1_score(y_all, p_all, average='macro'):.3f}")
print(f"mean conf   : {c_all.mean():.3f}   |   min conf: {c_all.min():.3f}")
print("\n" + classification_report(y_all, p_all, digits=3))

ConfusionMatrixDisplay(
    confusion_matrix(y_all, p_all, labels=classes),
    display_labels=classes).plot()
plt.title("CogniFusion — pooled CV confusion matrix")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(c_all, bins=25, color="steelblue", edgecolor="white")
ax.set_title("Prediction confidence distribution"); ax.set_xlabel("max softmax prob")
plt.tight_layout(); plt.show()

res = pd.DataFrame({"subject": pooled["subject"], "label": pooled["label"],
                    "epoch_id": pooled["epoch_id"], "pred": p_all,
                    "confidence": np.round(c_all, 3)})
for j, cname in enumerate(classes):
    res[f"p_{cname}"] = np.round(np.array(pooled["probs"])[:, j], 3)
res["correct"] = (res["label"] == res["pred"]).astype(int)
subj = res.groupby("subject")["correct"].mean().sort_values()
print("\nPer-subject accuracy (all 30 subjects, each predicted once):")
print(subj.round(3).to_string())
subj.plot.barh(figsize=(6, 5), color="seagreen")
plt.title("Accuracy per subject (pooled CV)"); plt.xlabel("accuracy")
plt.tight_layout(); plt.show()

# --- Sample predictions with full probability vectors ---
# recompute stacked probs per row is heavy; show pooled preds + confidence instead
res["confidence"] = np.round(c_all, 3)
display(res.head(15))

## 10 · Model diagnostics

Beyond accuracy: **ROC-AUC** (class separability), **calibration** (do the
probabilities mean what they say?), and **LightGBM feature importance**
(which fused features the tree model relies on most).

*Requires section 9 to have run (uses its pooled predictions).*

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.preprocessing import label_binarize

# pooled predictions from section 9
y_true = np.array(pooled["y"]); P = np.array(pooled["probs"])
classes_arr = np.array(classes)
Y = label_binarize(y_true, classes=classes_arr)

# ---------- ROC-AUC (one-vs-rest) ----------
auc = roc_auc_score(Y, P, average="macro", multi_class="ovr")
print(f"Macro ROC-AUC (ovr): {auc:.3f}")
for j, cname in enumerate(classes_arr):
    fpr, tpr, _ = roc_curve(Y[:, j], P[:, j])
    a = roc_auc_score(Y[:, j], P[:, j])
    print(f"  {cname:8s} AUC = {a:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
for j, cname in enumerate(classes_arr):
    fpr, tpr, _ = roc_curve(Y[:, j], P[:, j])
    ax.plot(fpr, tpr, label=f"{cname} (AUC={roc_auc_score(Y[:,j],P[:,j]):.3f})")
ax.plot([0,1],[0,1],"k--",lw=0.8); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC curves — one-vs-rest"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# ---------- Calibration (reliability diagram + ECE) ----------
conf = P.max(1); correct = (y_true == classes_arr[P.argmax(1)]).astype(float)
bins = np.linspace(0, 1, 11)
ece = 0.0
fig, ax = plt.subplots(figsize=(6, 5))
xs, ys = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf < hi)
    if m.sum() == 0: continue
    acc_bin, conf_bin = correct[m].mean(), conf[m].mean()
    xs.append(conf_bin); ys.append(acc_bin)
    ece += m.sum()/len(conf) * abs(acc_bin - conf_bin)
ax.plot([0,1],[0,1],"k--",lw=0.8,label="perfect")
ax.plot(xs, ys, "o-", color="steelblue", label=f"model (ECE={ece:.3f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy")
ax.set_title("Calibration — reliability diagram"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Expected Calibration Error: {ece:.3f}  |  Brier score: {brier_score_loss(y_true, P, labels=classes_arr):.3f}")

# ---------- LightGBM feature importance (averaged over folds) ----------
imp = np.zeros(len(eeg_cols) + len(eye_cols))
n_lgb = 0
colnames = list(eeg_cols) + list(eye_cols)
for fold in bundle["folds"]:
    if fold.get("lgb") is not None:
        imp += fold["lgb"].feature_importances_
        n_lgb += 1
if n_lgb:
    imp /= n_lgb
    order = np.argsort(imp)[::-1][:15]
    pd.Series(imp[order], index=[colnames[i] for i in order]).plot.barh(figsize=(9,5), color="seagreen")
    plt.title(f"Top-15 LightGBM feature importances (mean over {n_lgb} folds)")
    plt.xlabel("importance"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
else:
    print("No LightGBM models in checkpoint (train without --use-gbm to skip)")

# ---------- Error analysis: where does the model fail? ----------
res2 = pd.DataFrame({"y": y_true, "p": p_all})
res2["pred"] = classes_arr[P.argmax(1)]
res2["correct"] = (res2["y"] == res2["pred"]).astype(int)
acc_by_class = res2.groupby("y")["correct"].mean()
print("Accuracy by true class:")
print(acc_by_class.round(3).to_string())
acc_by_class.plot.bar(figsize=(5,3), color="steelblue")
plt.title("Per-class accuracy"); plt.ylabel("accuracy"); plt.ylim(0,1); plt.tight_layout(); plt.show()

# misclassified samples: show confidence distribution of errors
errs = res2[res2["correct"]==0]
print(f"Errors: {len(errs)}/{len(res2)} ({len(errs)/len(res2):.1%})")
if len(errs):
    print("Mean confidence on errors:", round(float(P[res2["correct"].values==0].max(1).mean()),3))
    print("Confusion pairs (true -> pred):")
    print(errs.groupby(["y","pred"]).size().to_string())


## 10b · Ablation — how much does each modality contribute?

Quick sklearn baselines on the **same subject-wise holdout**: RandomForest on
EEG-only, eye-only, and fused features. On synthetic data all three saturate —
the interesting comparison is on the **real dataset** (section 12), where eye
features typically dominate.

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from train import subject_wise_split, adaptive_subject_norm, coerce_numeric, make_norm_key

# works on whichever feature table was built last (synthetic or real)
df_a = pd.read_parquet("data/processed/features.parquet"
                       if Path("data/processed/features.parquet").exists()
                       else "data/processed/features_real.parquet")
df_a["norm_key"] = make_norm_key(df_a)
tr_a, te_a = subject_wise_split(df_a, test_frac=0.25, seed=42)
eeg_cols = [c for c in df_a.columns if c.startswith("eeg__")]
eye_cols = [c for c in df_a.columns if c.startswith("eye__")]

rows = []
for name, cols in [("EEG-only", eeg_cols), ("Eye-only", eye_cols), ("Fused", eeg_cols + eye_cols)]:
    Xtr = coerce_numeric(tr_a, cols); Xte = coerce_numeric(te_a, cols)
    trn, _ = adaptive_subject_norm(Xtr, tr_a["norm_key"])
    ten, _ = adaptive_subject_norm(Xte, te_a["norm_key"])
    m = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
    m.fit(trn.fillna(0), tr_a["label"])
    p = m.predict(ten.fillna(0))
    rows.append({"features": name, "n_cols": len(cols),
                 "accuracy": round(accuracy_score(te_a["label"], p), 4),
                 "macro_f1": round(f1_score(te_a["label"], p, average="macro"), 4)})
abl = pd.DataFrame(rows)
display(abl)
abl.set_index("features")[["accuracy","macro_f1"]].plot.bar(figsize=(7,4))
plt.title("Modality ablation — RandomForest baseline"); plt.ylabel("score"); plt.ylim(0,1)
plt.tight_layout(); plt.show()

## 11 · Data management & persistence

`src/data_management.py` is the same layer the live pipeline uses:

- **`DataLogger`** — SQLite database (`data/logs.db`) with `epochs`, `sessions`,
  `alerts` tables; `query_epochs()` / `get_statistics()` for analysis
- **`SessionManager`** — JSON session recordings under `sessions/` with
  epoch-level playback data

Below we log a sample of the held-out predictions exactly as the pipeline would,
then read them back. In Colab the DB is ephemeral — download `data/logs.db`
or export to CSV/JSON to keep it.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, "src")
from data_management import DataLogger, SessionManager

logger = DataLogger(log_dir="logs", db_path="data/logs.db")
sessions = SessionManager(storage_dir="sessions")

# --- record a "live" session from the pooled CV predictions ---
sid = sessions.create_session("colab_eval", {"source": "notebook", "model": "fusion_model.pt"})
# reuse section-9 results if they exist, else synthesize demo rows
try:
    sample = res.sample(min(30, len(res)), random_state=0)
except NameError:
    import pandas as _pd
    sample = _pd.DataFrame({
        "subject": ["S00"]*10 + ["S01"]*10 + ["S02"]*10,
        "label":   ["low"]*10 + ["medium"]*10 + ["high"]*10,
        "epoch_id": range(30),
        "pred":    ["low"]*10 + ["medium"]*10 + ["high"]*10,
        "confidence": [0.8]*30})

for _, r in sample.iterrows():
    epoch_data = {
        "timestamp": f"2026-11-22T10:{int(r['epoch_id'])//60:02d}:{int(r['epoch_id'])%60:02d}",
        "prediction": r["pred"],
        "confidence": float(r["confidence"]),
        "eeg_trust": 0.52, "eye_trust": 0.48,
        "processing_time": 0.12,
        "features": {"note": f"subject={r['subject']}", "true_label": r["label"]},
    }
    sessions.add_epoch_to_session(sid, epoch_data)
    logger.log_epoch(epoch_data, session_id=sid)

sessions.close_session(sid)
logger.log_session({
    "session_id": sid, "user_id": "colab_eval",
    "start_time": "2026-11-22T10:00:00", "end_time": "2026-11-22T10:30:00",
    "epoch_count": len(sample), "metadata": {"source": "notebook"},
})
logger.log_alert({"type": "info", "severity": "low",
                  "message": "Notebook evaluation session logged"}, session_id=sid)

print("=== Database contents ===")
db = logger.query_epochs()
print(f"{len(db)} epoch rows in SQLite")
display(db[["timestamp", "prediction", "confidence", "eeg_trust", "eye_trust"]].head(10))
print("Statistics:", json.dumps(logger.get_statistics(), indent=2))

# --- session playback file ---
rec = sessions.get_session(sid)
print("Session recording:", rec["session_id"], "-", rec["epoch_count"], "epochs, status:", rec["status"])
Path("sessions").mkdir(exist_ok=True)


In [ ]:
# Export the logged epochs to CSV / JSON for reuse outside Colab
out = logger.query_epochs()
out.to_csv("data/processed/eval_epochs.csv", index=False)
out.to_json("data/processed/eval_epochs.json", orient="records", indent=2)
print("Exported:", "data/processed/eval_epochs.csv", "and .json")

# Optional: download the SQLite DB itself in Colab
try:
    from google.colab import files
    files.download("data/logs.db")
    print("logs.db download started")
except ImportError:
    print("Not in Colab — DB saved at data/logs.db")

# ---------- Session-log timeline: confidence over logged epochs ----------
import matplotlib.pyplot as plt
db = logger.query_epochs().reset_index(drop=True)
if len(db):
    fig, ax = plt.subplots(figsize=(10,3))
    colors = {"low":"seagreen","medium":"goldenrod","high":"tomato"}
    for i,(idx,row) in enumerate(db.iterrows()):
        ax.scatter(idx, row["confidence"], c=colors.get(row["prediction"],"gray"), s=30)
    ax.set_title("Logged session — prediction confidence over epochs")
    ax.set_xlabel("epoch #"); ax.set_ylabel("confidence")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],marker="o",ls="",color=colors[k],label=k) for k in colors], fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"Plotted {len(db)} logged epochs")


## 12 · Real dataset (optional) — 113 participants

The synthetic pipeline above is for demo. For the **real benchmark**, download the
public *Consumer-Grade EEG-based Eye Tracking* dataset (Zenodo, 1.5 GB CSV:
Muse EEG @256 Hz + webcam gaze, 116 sessions, saccade/smooth-pursuit tasks at two
difficulty levels) and rebuild features from it.

> ⏳ Feature extraction over ~10k real epochs takes **hours on CPU** — run this
> section only if you have time, or run it once and reuse `features_real.parquet`.

In [ ]:
# Download the real dataset (~1.5 GB) from Zenodo
import os
if not os.path.exists("data/raw/dataset.csv"):
    !wget -q --show-progress "https://zenodo.org/api/records/14860668/files/dataset.csv/content" -O data/raw/dataset.csv
else:
    print("dataset.csv already present")
!ls -lh data/raw/dataset.csv

In [ ]:
# Build fused features from real data (long step — prints progress every 1000 epochs)
import sys; sys.path.insert(0, "src")
from pathlib import Path
FEAT_REAL = Path("data/processed/features_real.parquet")
if FEAT_REAL.exists():
    print("features_real.parquet exists — skipping")
    import pandas as pd
    df_real = pd.read_parquet(FEAT_REAL)
else:
    from build_dataset_real import build
    df_real = build()
df_real.shape

In [ ]:
# Train on the real dataset — subject-wise 5-fold CV for honest generalization
!python src/train.py --features data/processed/features_real.parquet                     --epochs 200 --cv 5 --use-gbm                     --out models/fusion_model_real

## 13 · Next steps

- **Real data**: `python src/build_dataset_real.py` builds the fused table from the
  1.5 GB real Muse-EEG + webcam dataset (`data/raw/dataset.csv`, 113 participants —
  see the repo README), then train with
  `--features data/processed/features_real.parquet --cv 5`.
- **Live demo**: `python src/realtime_pipeline.py --eeg synthetic --eye synthetic`
  locally, or `streamlit run dashboard.py` for the monitoring dashboard. Webcam
  eye-tracking works in Colab (`src/webcam_eyetracker.py`); EEG needs Muse
  hardware (BrainFlow).
- **Full training**: `--epochs 300 --cv 5` for the production-grade subject-wise CV.

## Limitations

- Synthetic data is generated from known class-conditional patterns — scores here
  are an upper bound, not a real-world guarantee.
- `medium` is intentionally the ambiguous class (partial overlap in the generator);
  expect lower F1 there than on `low`/`high`.
- Single-epoch inference in the notebook uses the holdout split; live deployment
  uses session-adaptive calibration over a rolling window instead.

## 15 · Notebook power-ups

Quick-win extras: sklearn baseline grid, interactive widgets, interpretability
snapshots, live-stream simulation, exports, and a final integrity checklist.

In [ ]:
# Baseline grid — how does the NN ensemble compare to classic ML?
import sys; sys.path.insert(0,"src"); import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from train import subject_wise_split, adaptive_subject_norm, coerce_numeric, make_norm_key

df_b = pd.read_parquet("data/processed/features.parquet")
df_b["norm_key"]=make_norm_key(df_b)
tr_b,te_b=subject_wise_split(df_b,test_frac=0.25,seed=42)
cols=[c for c in df_b.columns if c.startswith(("eeg__","eye__"))]
Xtr=coerce_numeric(tr_b,cols); Xte=coerce_numeric(te_b,cols)
trn,_=adaptive_subject_norm(Xtr,tr_b["norm_key"]); ten,_=adaptive_subject_norm(Xte,te_b["norm_key"])
Xtr_f=trn.fillna(0); Xte_f=ten.fillna(0); ytr=tr_b["label"]; yte=te_b["label"]
models={"LogReg":LogisticRegression(max_iter=2000),"RF":RandomForestClassifier(300,random_state=0,n_jobs=-1),
        "GBM":GradientBoostingClassifier(random_state=0),"SVM":SVC(probability=True),"kNN":KNeighborsClassifier(7)}
rows=[]
for n,m in models.items():
    m.fit(Xtr_f,ytr); p=m.predict(Xte_f)
    rows.append({"model":n,"acc":round(accuracy_score(yte,p),3),"f1":round(f1_score(yte,p,average="macro"),3)})
display(pd.DataFrame(rows).sort_values("f1",ascending=False))

In [ ]:
# Interactive class explorer
try:
    from ipywidgets import interact, widgets
    feat_cols=[c for c in df.columns if c.startswith(("eeg__","eye__"))]
    @interact(feature=widgets.Dropdown(options=sorted(feat_cols)[:40],value=sorted(feat_cols)[0]),
              cls=widgets.Dropdown(options=["low","medium","high","all"],value="all"))
    def _peek(feature,cls):
        d=df if cls=="all" else df[df["label"]==cls]
        d[feature].hist(bins=40); plt.title(f"{feature} — {cls}"); plt.show()
except ImportError:
    print("ipywidgets not installed — pip install ipywidgets")

In [ ]:
# EEG-vs-eye trust per prediction (gate weights from the NN)
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
ckpt=torch.load("models/fusion_model.pt",map_location="cpu",weights_only=False)
rec=ckpt["folds"][0]
from model import AttentionFusionNet
m=AttentionFusionNet(eeg_dim=len(ckpt["eeg_cols"]),eye_dim=len(ckpt["eye_cols"]),
    embed_dim=rec.get("embed_dim",48),n_classes=len(ckpt["label_classes"]),gate_type="vector")
m.load_state_dict(rec["model_state"]); m.eval()
# reuse pooled test features
te_feats = pd.read_parquet("data/processed/features.parquet")
te_feats["norm_key"]=make_norm_key(te_feats)
_,te=subject_wise_split(te_feats,test_frac=0.25,seed=42)
eeg_n,_=adaptive_subject_norm(coerce_numeric(te,ckpt["eeg_cols"]),te["norm_key"])
eye_n,_=adaptive_subject_norm(coerce_numeric(te,ckpt["eye_cols"]),te["norm_key"])
xe=np.nan_to_num(rec["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(rec["scaler_eye"].transform(eye_n))
with torch.no_grad():
    logits,w=m(torch.tensor(xe,dtype=torch.float32),torch.tensor(ye,dtype=torch.float32),return_weight=True)
trust=pd.DataFrame({"eeg_trust":w[:,0].numpy(),"eye_trust":1-w[:,0].numpy(),"label":te["label"].values})
trust.groupby("label")[["eeg_trust","eye_trust"]].mean().plot.bar(figsize=(6,3))
plt.title("Mean modality trust by class (vector gate)"); plt.ylabel("trust weight"); plt.ylim(0,1)
plt.tight_layout(); plt.show()
print(trust.groupby("label")[["eeg_trust","eye_trust"]].mean().round(3))

In [ ]:
# Simulated live stream — rolling predictions over the test set
import numpy as np, pandas as pd, matplotlib.pyplot as plt
# reuse `probs`/`pred`/`y_true` from section 9 if present, else skip
try:
    order=np.argsort(pooled["epoch_id"]); yy=np.array(pooled["y"])[order]; pp=np.array(pooled["pred"])[order]
    conf=np.array(pooled["conf"])[order]
    fig,ax=plt.subplots(figsize=(11,3))
    colors={"low":"seagreen","medium":"goldenrod","high":"tomato"}
    for i in range(len(yy)):
        ax.scatter(i,conf[i],c=colors.get(yy[i],"gray"),s=25)
    ax.set_title("Simulated live stream — confidence over epochs (color=true label)")
    ax.set_xlabel("epoch index"); ax.set_ylabel("confidence")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],marker="o",ls="",color=colors[k],label=k) for k in colors],fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"{len(yy)} epochs replayed; accuracy={np.mean(yy==pp):.1%}")
except NameError:
    print("Run section 9 first")

In [ ]:
# Export bundle: checkpoint + metrics + predictions in one zip
import json as _json, torch, pandas as pd, os
os.makedirs("exports",exist_ok=True)
summary={"cv":ckpt["cv_summary"],"n_folds":len(ckpt["folds"])}
open("exports/metrics.json","w").write(_json.dumps(summary,indent=2))
try:
    res.to_csv("exports/predictions.csv",index=False)
except NameError:
    pass
try:
    pd.DataFrame(pooled["probs"]).to_csv("exports/probabilities.csv",index=False)
except NameError:
    pass
print("exports/:",os.listdir("exports"))
from google.colab import files
for f in os.listdir("exports"): files.download(f"exports/{f}")
print("downloads started")

In [ ]:
# Integrity checklist — verify every artifact exists before you leave
from pathlib import Path
checks={
 "features.parquet":Path("data/processed/features.parquet").exists(),
 "model checkpoint":Path("models/fusion_model.pt").exists(),
 "logs.db":Path("data/logs.db").exists(),
 "sessions dir":Path("sessions").exists(),
}
for k,v in checks.items(): print(f"  {'OK ' if v else 'MISS'} {k}")
print("All green" if all(checks.values()) else "Some artifacts missing — rerun earlier sections")

## 14 · Run summary

Compact recap of everything the notebook produced.

In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print("="*55)
print("COGNIFUSION — NOTEBOOK RUN SUMMARY")
print("="*55)
print(f"Dataset      : {df.shape[0]} epochs, {df['subject'].nunique()} subjects, "
      f"{df['label'].nunique()} classes ({len(df.columns)-3} features)")
print(f"Model        : {len(ckpt['folds'])}-fold ensemble "
      f"({'NN+GBM' if ckpt['folds'][0].get('lgb_model') else 'NN-only'})")
print(f"CV accuracy  : {ckpt['cv_summary']['mean_accuracy']:.1%} "
      f"(macro-F1 {ckpt['cv_summary']['mean_macro_f1']:.3f})")
try:
    print(f"Pooled eval  : {accuracy_score(y_true, p_all):.1%} on "
          f"{len(y_true)} held-out epochs")
    print(f"ROC-AUC      : {auc:.3f}   ECE: {ece:.3f}")
except NameError:
    print("Pooled eval  : (run sections 9-10 first)")
try:
    print(f"DB logged    : {logger.get_statistics()}")
except NameError:
    print("DB logged    : (run section 11 first)")
print("="*55)

## 16 · Final extras

Seed stability, confidence-threshold sweep, and model export.

In [ ]:
# Seed stability — retrain quickly at 2 seeds, compare holdout accuracy
# (uses fewer epochs for speed; production runs use 200)
import subprocess, sys
for seed in (1, 2):
    !python src/train.py --features data/processed/features.parquet --epochs 40 --cv 0 --seed {seed} --out /tmp/fm_seed{seed} 2>/dev/null | tail -3
import torch
for seed in (1,2):
    c=torch.load(f"/tmp/fm_seed{seed}.pt",map_location="cpu",weights_only=False)
    print(f"seed {seed}: acc={c['cv_summary']['mean_accuracy']:.3f} f1={c['cv_summary']['mean_macro_f1']:.3f}")

In [ ]:
# Confidence-threshold sweep — accuracy vs coverage
try:
    import numpy as np, pandas as pd, matplotlib.pyplot as plt
    conf=np.array(pooled["conf"]); correct=(np.array(pooled["y"])==np.array(pooled["pred"])).astype(float)
    ths=np.linspace(0.3,0.95,14); accs=[];covs=[]
    for t in ths:
        m=conf>=t; covs.append(m.mean()); accs.append(correct[m].mean() if m.sum() else 0)
    plt.plot(covs,accs,"o-"); plt.xlabel("coverage"); plt.ylabel("accuracy on covered")
    plt.title("Confidence threshold sweep"); plt.grid(alpha=.3); plt.show()
except NameError:
    print("Run section 9 first")

In [ ]:
# Export the ensemble to ONNX (single fold) for deployment
try:
    import torch
    rec=ckpt["folds"][0]
    from model import AttentionFusionNet
    m=AttentionFusionNet(eeg_dim=len(ckpt["eeg_cols"]),eye_dim=len(ckpt["eye_cols"]),
        embed_dim=rec.get("embed_dim",48),n_classes=len(ckpt["label_classes"]),gate_type="vector")
    m.load_state_dict(rec["model_state"]); m.eval()
    xe=torch.randn(1,len(ckpt["eeg_cols"])); ye=torch.randn(1,len(ckpt["eye_cols"]))
    torch.onnx.export(m,(xe,ye),"exports/fusion_fold0.onnx",
        input_names=["eeg","eye"],output_names=["logits"],opset_version=17)
    print("exported exports/fusion_fold0.onnx")
    from google.colab import files; files.download("exports/fusion_fold0.onnx")
except Exception as e:
    print("ONNX export skipped:",e)

## 17 · Experiments

**Hyperparameter sweep** — small grid over epochs/lr/embed via quick runs.
**LOSO** — `--cv 30` = leave-one-subject-out (each of 30 subjects held out once);
the strictest generalization test. Both write checkpoints under `/tmp` so your
main `models/fusion_model.pt` is untouched.

In [ ]:
# Sweep: 4 quick configs x 40 epochs, single holdout — collect accuracy
import subprocess, json as _json, torch, itertools, warnings
warnings.filterwarnings("ignore")
configs=[
    {"epochs":40,"lr":1e-3,"embed":48},
    {"epochs":40,"lr":5e-4,"embed":48},
    {"epochs":40,"lr":1e-3,"embed":96},
    {"epochs":40,"lr":2e-3,"embed":32},
]
rows=[]
for i,cfg in enumerate(configs):
    out=f"/tmp/sweep_{i}.pt"
    r=subprocess.run([sys.executable,"src/train.py","--features","data/processed/features.parquet",
        "--epochs",str(cfg["epochs"]),"--cv","0","--lr",str(cfg["lr"]),
        "--embed-dim",str(cfg["embed"]),"--out",out],
        capture_output=True,text=True,cwd=".")
    try:
        ck=torch.load(out,map_location="cpu",weights_only=False)
        acc=ck["cv_summary"]["mean_accuracy"]; f1=ck["cv_summary"]["mean_macro_f1"]
        rows.append({**cfg,"acc":round(acc,3),"f1":round(f1,3)})
        print(f"cfg{i}: acc={acc:.3f} f1={f1:.3f}")
    except Exception as e:
        print(f"cfg{i} failed:",r.stderr[-300:] if r.stderr else e)
sw=pd.DataFrame(rows); display(sw)
sw.to_csv("exports/sweep.csv",index=False)

In [ ]:
# LOSO — leave-one-subject-out (30 folds, slower; drop epochs for a quick pass)
# python src/train.py --features data/processed/features.parquet --epochs 60 --cv 30 --use-gbm --out /tmp/fm_loso
!python src/train.py --features data/processed/features.parquet --epochs 60 --cv 30 --use-gbm --out /tmp/fm_loso 2>/dev/null | tail -15
import torch, warnings; warnings.filterwarnings("ignore")
try:
    ck=torch.load("/tmp/fm_loso.pt",map_location="cpu",weights_only=False)
    print("LOSO summary:",ck["cv_summary"])
    accs=[f.get("val_accuracy") for f in ck["folds"]]
    accs=[a for a in accs if a is not None]
    print(f"LOSO per-subject acc: mean={np.mean(accs):.3f} min={np.min(accs):.3f} max={np.max(accs):.3f} n={len(accs)}")
except Exception as e:
    print("LOSO checkpoint not found —",e)

## 17b · Statistical rigor

Bootstrap confidence intervals on the pooled score — how much could the
headline accuracy move if we resampled the test epochs?

In [ ]:
try:
    import numpy as np
    y=np.array(pooled["y"]); pr=np.array(pooled["pred"]); conf=np.array(pooled["conf"])
    rng=np.random.default_rng(42); n=len(y); B=2000
    accs=np.empty(B); f1s=np.empty(B)
    from sklearn.metrics import f1_score
    for b in range(B):
        idx=rng.integers(0,n,n)
        accs[b]=(y[idx]==pr[idx]).mean()
        f1s[b]=f1_score(y[idx],pr[idx],average="macro")
    print(f"Pooled accuracy : {accs.mean():.3f}  95% CI [{np.percentile(accs,2.5):.3f}, {np.percentile(accs,97.5):.3f}]")
    print(f"Pooled macro-F1 : {f1s.mean():.3f}  95% CI [{np.percentile(f1s,2.5):.3f}, {np.percentile(f1s,97.5):.3f}]")
    # vs majority-class baseline
    maj=max(pd.Series(y).value_counts(normalize=True))
    print(f"Majority-class baseline accuracy: {maj:.3f}  (model beats it by {(accs.mean()-maj)*100:.1f} pts)")
except NameError:
    print("Run section 9 first")

## 17c · Sequence model (GRU)

`--seq-len N` groups N consecutive epochs per subject into a window; the model's
GRU path encodes temporal context. Verified end-to-end (train → checkpoint →
live pipeline). Use `seq_len=3` for ~4s×3=12s context.

In [ ]:
# Train a sequence-aware model (slower — windows reduce sample count)
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 5 --seq-len 3 --out models/fusion_model_seq
# then point the live pipeline at it:
# python src/realtime_pipeline.py --model models/fusion_model_seq.pt --eeg synthetic --eye synthetic

## 17d · Neural modality baselines

`--ablate eeg|eye` zeroes one modality before training — same architecture,
one dead input channel — giving true single-modality neural nets to compare
against the fused model (and each other). GBM auto-disables for clean NN-only
comparison. Checkpoints record `"ablate"` in each fold.

In [ ]:
# EEG-only neural baseline (eye input zeroed)
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 5 --ablate eye --out models/fusion_eeg_only

# Eye-only neural baseline (EEG input zeroed)
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 5 --ablate eeg --out models/fusion_eye_only

# Compare all three checkpoints' cv_summary after training:
import torch
for name in ["fusion_model","fusion_eeg_only","fusion_eye_only"]:
    try:
        c=torch.load(f"models/{name}.pt",map_location="cpu",weights_only=False)
        s=c["cv_summary"]
        print(f"{name:20s} acc={s['mean_accuracy']:.3f}  f1={s['mean_macro_f1']:.3f}")
    except Exception as e:
        print(name,"->",e)

## 17e · Model comparison board

Loads every checkpoint variant (fused / EEG-only / eye-only / sequence) and
prints a single comparison table — the ablation story at a glance.

In [ ]:
import torch, pandas as pd, glob
rows=[]
for path in sorted(glob.glob("models/fusion*.pt")):
    try:
        c=torch.load(path,map_location="cpu",weights_only=False)
        s=c.get("cv_summary",{})
        f0=c["folds"][0]
        rows.append({
            "checkpoint": path.split("/")[-1].replace(".pt",""),
            "folds": len(c["folds"]),
            "seq_len": f0.get("seq_len",1),
            "ablate": f0.get("ablate") or "-",
            "gbm": "yes" if f0.get("lgb_model") is not None else "no",
            "acc": round(s.get("mean_accuracy",float("nan")),3),
            "f1":  round(s.get("mean_macro_f1",float("nan")),3),
            "acc_std": round(s.get("std_accuracy",float("nan")),3),
        })
    except Exception as e:
        print(path,"->",e)
board=pd.DataFrame(rows).sort_values("f1",ascending=False)
display(board)
board.to_csv("exports/model_comparison.csv",index=False)
print("saved -> exports/model_comparison.csv")

## 17f · Interpretability — SHAP on the LightGBM stack

TreeSHAP contributions from the fold LightGBM models: which fused features
push predictions toward each class. Requires a `--use-gbm` checkpoint.

In [ ]:
try:
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, warnings
    warnings.filterwarnings("ignore")
    # rebuild one fold's scaled test features (same path as section 9)
    bundle=load_ensemble("models/fusion_model.pt")
    fold=bundle["folds"][0]
    df_s=pd.read_parquet("data/processed/features.parquet"); df_s["norm_key"]=make_norm_key(df_s)
    k=len(bundle["folds"]); splits=list(subject_kfold(df_s,k=k,seed=42)) if k>1 else [subject_wise_split(df_s,test_frac=0.25,seed=42)]
    _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    X=np.concatenate([xe,ye],axis=1); colnames=list(eeg_cols)+list(eye_cols)
    if fold.get("lgb") is not None:
        contrib=fold["lgb"].predict(X, pred_contrib=True)
        # multiclass SHAP: (n, n_classes*(n_features+1)) -> reshape
        ncls=len(classes); nf=X.shape[1]+1
        c3=contrib.reshape(len(X),ncls,nf)
        imp=np.abs(c3).mean(axis=(0,1))[:nf-1]  # drop bias term
        order=np.argsort(imp)[::-1][:15]
        pd.Series(imp[order],index=[colnames[i] for i in order]).plot.barh(figsize=(9,5),color="darkorange")
        plt.title("Top-15 mean |SHAP| feature contributions (fold-1 LightGBM)")
        plt.xlabel("mean |contribution|"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
    else:
        print("No LightGBM in checkpoint — train with --use-gbm")
except NameError as e:
    print("Run sections 7+9 first:",e)

## 18 · Advanced evaluation

Split-conformal prediction sets (distribution-free coverage guarantees),
per-fold ROC-AUC, risk-coverage curves, and permutation importance on the NN.

In [ ]:
try:
    import numpy as np
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import label_binarize
    # --- per-fold ROC-AUC: recompute each fold's probs on its own split ---
    y_all=[]; P_all=[]; fold_id=[]
    with torch.no_grad():
        for i,fold in enumerate(bundle["folds"]):
            _,te=splits[i]; te=te.copy(); te["norm_key"]=make_norm_key(te)
            eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
            eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
            xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
            logits=fold["model"](torch.tensor(xe,dtype=torch.float32),torch.tensor(ye,dtype=torch.float32))
            logits=fold["temp"](logits); p=torch.softmax(logits,dim=1).numpy()
            if fold.get("lgb") is not None:
                lp=fold["lgb"].predict_proba(np.concatenate([xe,ye],axis=1))
                p=0.6*p+0.4*lp
            y_all+=list(te["label"]); P_all+=list(p); fold_id+=[i]*len(te)
    y_arr=np.array(y_all); P_arr=np.array(P_all); fold_arr=np.array(fold_id)
    Y=label_binarize(y_arr,classes=np.array(classes))
    aucs=[]
    for i in range(len(bundle["folds"])):
        m=fold_arr==i
        aucs.append(roc_auc_score(Y[m],P_arr[m],average="macro",multi_class="ovr"))
    print("Per-fold ROC-AUC:",[round(a,3) for a in aucs],f"mean={np.mean(aucs):.3f}")
except NameError as e:
    print("Run section 9 first:",e)

In [ ]:
try:
    # --- split-conformal prediction sets (coverage guarantee) ---
    import numpy as np
    rng=np.random.default_rng(0); n=len(y_arr)
    idx=rng.permutation(n); cal, tst = idx[:n//2], idx[n//2:]
    # nonconformity = 1 - p_true
    scores=1-P_arr[np.arange(n),[classes.index(t) for t in y_arr]]
    q=np.quantile(scores[cal],0.9)  # 90% coverage target
    sets=[]
    for p in P_arr[tst]:
        sets.append(set(np.array(classes)[p>=1-q]))
    covered=np.mean([y_arr[t] in s for t,s in zip(tst,sets)])
    sizes=np.mean([len(s) for s in sets])
    print(f"Conformal @90% target -> empirical coverage {covered:.1%}, mean set size {sizes:.2f}")
    # --- risk-coverage curve ---
    import matplotlib.pyplot as plt
    conf=P_arr.max(1); correct=(y_arr==np.array(classes)[P_arr.argmax(1)]).astype(float)
    order=np.argsort(-conf); risks=1-np.cumsum(correct[order])/np.arange(1,n+1)
    plt.plot(np.arange(1,n+1)/n,risks); plt.xlabel("coverage"); plt.ylabel("risk (1-acc)")
    plt.title("Risk-coverage curve"); plt.grid(alpha=.3); plt.show()
except NameError as e:
    print("Run the ROC-AUC cell first:",e)

In [ ]:
try:
    # --- permutation importance on NN probs (modality groups) ---
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, warnings
    warnings.filterwarnings("ignore")
    fold=bundle["folds"][0]; _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    X=np.concatenate([xe,ye],axis=1); yt=te["label"].values
    def acc_of(Xm):
        with torch.no_grad():
            lo=fold["model"](torch.tensor(Xm[:,:len(eeg_cols)]),torch.tensor(Xm[:,len(eeg_cols):]))
            lo=fold["temp"](lo); pr=np.array(classes)[torch.softmax(lo,1).numpy().argmax(1)]
        return (pr==yt).mean()
    base=acc_of(X); rng=np.random.default_rng(0)
    drops=[]
    for name,sl in [("EEG block",slice(0,len(eeg_cols))),("eye block",slice(len(eeg_cols),None))]:
        Xp=X.copy(); Xp[:,sl]=Xp[rng.permutation(len(Xp))][:,sl]
        drops.append((name,round(base-acc_of(Xp),3)))
    print("base acc:",round(base,3),"| permutation drops:",drops)
except NameError as e:
    print("Run section 9 first:",e)

In [ ]:
# seq_len sweep — which temporal window is best? (quick 60-epoch runs)
for sl in (1,2,3,5):
    !python src/train.py --features data/processed/features.parquet --epochs 60 --cv 0 --seq-len {sl} --out /tmp/fm_sl{sl} 2>/dev/null | tail -2
import torch, warnings; warnings.filterwarnings("ignore")
for sl in (1,2,3,5):
    try:
        c=torch.load(f"/tmp/fm_sl{sl}.pt",map_location="cpu",weights_only=False)
        print(f"seq_len={sl}: acc={c['cv_summary']['mean_accuracy']:.3f} f1={c['cv_summary']['mean_macro_f1']:.3f}")
    except Exception as e: print(f"seq_len={sl}: {e}")

## 18b · Ordinal metrics & per-class calibration

Load is ordinal (low < medium < high): quadratic-weighted kappa and
ordinal-MAE reward near-misses. Per-class ECE shows which class is
miscalibrated.

In [ ]:
try:
    import numpy as np
    from sklearn.metrics import cohen_kappa_score, mean_absolute_error
    from sklearn.preprocessing import LabelEncoder
    yi=np.array([classes.index(t) for t in y_arr]); pi=np.array([classes.index(t) for t in np.array(classes)[P_arr.argmax(1)]])
    qwk=cohen_kappa_score(yi,pi,weights="quadratic")
    print(f"Quadratic-weighted kappa: {qwk:.3f}   |   ordinal MAE: {mean_absolute_error(yi,pi):.3f} class-steps")
    # per-class ECE
    pred=np.array(classes)[P_arr.argmax(1)]; conf=P_arr.max(1); correct=(pred==y_arr).astype(float)
    for cname in classes:
        m=pred==cname
        if m.sum()<5: continue
        bins=np.linspace(0,1,6); ece=0
        for lo,hi in zip(bins[:-1],bins[1:]):
            mm=m&(conf>=lo)&(conf<hi)
            if mm.sum(): ece+=mm.sum()/m.sum()*abs(correct[mm].mean()-conf[mm].mean())
        print(f"  {cname:8s} n={m.sum():4d}  ECE={ece:.3f}  acc={correct[m].mean():.3f}")
except NameError as e:
    print("Run section 18 first:",e)

In [ ]:
# Deep ensemble: train 3 seeds (60 epochs each) and average holdout probs
import subprocess, sys, torch, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0,"src")
from train import subject_wise_split, adaptive_subject_norm, coerce_numeric, make_norm_key, TemperatureScaler
from model import AttentionFusionNet

df_e=pd.read_parquet("data/processed/features.parquet"); df_e["norm_key"]=make_norm_key(df_e)
tr_e,te_e=subject_wise_split(df_e,test_frac=0.25,seed=42)
eeg_c=[c for c in df_e if c.startswith("eeg__")]; eye_c=[c for c in df_e if c.startswith("eye__")]
P_seeds=[]; yt_e=te_e["label"].values
for seed in (1,2,3):
    out=f"/tmp/de_{seed}.pt"
    subprocess.run([sys.executable,"src/train.py","--features","data/processed/features.parquet",
        "--epochs","60","--cv","0","--seed",str(seed),"--out",out],
        capture_output=True,text=True)
    ck=torch.load(out,map_location="cpu",weights_only=False); rec=ck["folds"][0]
    m=AttentionFusionNet(eeg_dim=len(eeg_c),eye_dim=len(eye_c),embed_dim=rec.get("embed_dim",48),
        n_classes=3,gate_type="vector",seq_len=rec.get("seq_len",1))
    m.load_state_dict(rec["model_state"]); m.eval()
    ts=TemperatureScaler(); ts.load_state_dict(rec["temp_scaler"]); ts.eval()
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te_e,eeg_c),te_e["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te_e,eye_c),te_e["norm_key"])
    xe=np.nan_to_num(rec["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(rec["scaler_eye"].transform(eye_n))
    with torch.no_grad():
        lo=ts(m(torch.tensor(xe,dtype=torch.float32),torch.tensor(ye,dtype=torch.float32)))
        P_seeds.append(torch.softmax(lo,1).numpy())
from sklearn.metrics import accuracy_score, f1_score
P_avg=np.mean(P_seeds,axis=0)
pred=np.array(ck["label_classes"])[P_avg.argmax(1)]
print(f"Deep ensemble (3 seeds): acc={accuracy_score(yt_e,pred):.3f} f1={f1_score(yt_e,pred,average='macro'):.3f}")
for i,s in enumerate(P_seeds):
    pr=np.array(ck["label_classes"])[s.argmax(1)]
    print(f"  seed{i+1} alone: acc={accuracy_score(yt_e,pr):.3f}")

## 18c · Uncertainty & adaptation

**MC-dropout** — stochastic forward passes give predictive uncertainty;
high-uncertainty epochs should be the mistakes. **Few-shot adaptation** —
fine-tune only the classifier head on k labeled epochs from an unseen
subject to measure calibration gain.

In [ ]:
try:
    # --- MC-dropout: enable ONLY Dropout modules, keep model.eval() elsewhere ---
    import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
    fold=bundle["folds"][0]
    _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    Xt=torch.tensor(xe,dtype=torch.float32); Yt=torch.tensor(ye,dtype=torch.float32)
    m=fold["model"]; m.eval()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.train()
    probs=[]
    with torch.no_grad():
        for _ in range(30):
            lo=fold["temp"](m(Xt,Yt)); probs.append(torch.softmax(lo,1).numpy())
    P_mc=np.stack(probs)              # (30, n, 3)
    mean_p=P_mc.mean(0); std_p=P_mc.std(0).max(1)   # predictive uncertainty
    pred=mean_p.argmax(1); yt=te["label"].values
    acc=(np.array(classes)[pred]==yt)
    plt.scatter(std_p[~acc],mean_p[~acc].max(1)[~acc],c="red",s=18,label="errors")
    plt.scatter(std_p[acc],mean_p[acc].max(1)[acc],c="seagreen",s=18,label="correct",alpha=.6)
    plt.xlabel("predictive std (MC-dropout)"); plt.ylabel("confidence")
    plt.title("Uncertainty vs correctness"); plt.legend(fontsize=8); plt.show()
    print(f"mean uncertainty on errors={std_p[~acc].mean():.3f} vs correct={std_p[acc].mean():.3f}")
except NameError as e:
    print("Run section 9 first:",e)

In [ ]:
try:
    # --- Few-shot subject adaptation: fine-tune classifier head on k epochs ---
    import torch, torch.nn as nn, numpy as np, copy
    from sklearn.metrics import accuracy_score
    fold=bundle["folds"][0]
    # pick the worst subject from section-9 pooled eval
    try:
        subj_acc=res.groupby("subject")["correct"].mean()
        target=subj_acc.idxmin()
    except NameError:
        target="S00"
    _,te0=splits[0]; te0=te0.copy(); te0["norm_key"]=make_norm_key(te0)
    te_subj=te0[te0["subject"]==target].head(4)   # k=4 labeled epochs
    rest=te0[(te0["subject"]==target)].iloc[4:] or te0[te0["subject"]!=target]
    rest=te0[te0["subject"]==target].iloc[4:]
    if len(rest)==0: rest=te0[te0["subject"]!=target]

    m2=copy.deepcopy(fold["model"]); m2.eval()
    for p in m2.parameters(): p.requires_grad=False
    for p in m2.classifier.parameters(): p.requires_grad=True
    opt=torch.optim.AdamW(m2.classifier.parameters(),lr=1e-3)

    def feats(d):
        d=d.copy(); d["norm_key"]=make_norm_key(d)
        en,_=adaptive_subject_norm(coerce_numeric(d,eeg_cols),d["norm_key"])
        yn,_=adaptive_subject_norm(coerce_numeric(d,eye_cols),d["norm_key"])
        return (torch.tensor(np.nan_to_num(fold["scaler_eeg"].transform(en)),dtype=torch.float32),
                torch.tensor(np.nan_to_num(fold["scaler_eye"].transform(yn)),dtype=torch.float32))

    xs_e,xs_y=feats(te_subj)
    ys=torch.tensor([classes.index(v) for v in te_subj["label"]],dtype=torch.long)
    m2.train()
    for _ in range(60):
        opt.zero_grad(); loss=torch.nn.functional.cross_entropy(m2(xs_e,xs_y),ys)
        loss.backward(); opt.step()
    m2.eval()
    xe_r,ye_r=feats(rest); yt_r=rest["label"].values
    with torch.no_grad():
        p_before=np.array(classes)[torch.softmax(fold["temp"](fold["model"](xe_r,ye_r)),1).numpy().argmax(1)]
        p_after =np.array(classes)[torch.softmax(m2(xe_r,ye_r),1).numpy().argmax(1)]
    print(f"Subject {target}: acc before={accuracy_score(yt_r,p_before):.3f} -> after 4-epoch head adaptation={accuracy_score(yt_r,p_after):.3f}")
except NameError as e:
    print("Run section 9 first:",e)
except Exception as e:
    print("adaptation demo error:",type(e).__name__,e)


## 18d · Per-class conformal sets

Global conformal treats all classes equally; per-class calibration gives each
class its own coverage guarantee — useful when `medium` is inherently ambiguous.

In [ ]:
try:
    import numpy as np
    rng=np.random.default_rng(0); n=len(y_arr)
    idx=rng.permutation(n); cal,tst=idx[:n//2],idx[n//2:]
    cls_arr=np.array(classes)
    # per-class nonconformity quantiles
    q={}
    for cname in classes:
        cm=cal[[classes.index(y_arr[i])==classes.index(cname) for i in cal]]
        # scores for calibration points whose TRUE class is cname
        mask_cal=np.array([y_arr[i]==cname for i in cal])
        if mask_cal.sum()<10: q[cname]=0.5; continue
        sc=1-P_arr[cal][mask_cal,[classes.index(cname)]]
        q[cname]=np.quantile(sc,0.9)
    covered=[]; sizes=[]
    for t in tst:
        p=P_arr[t]; s=set(cls_arr[p>=1-np.array([q[c] for c in classes])])
        covered.append(y_arr[t] in s); sizes.append(len(s))
    print(f"Per-class conformal @90%: coverage={np.mean(covered):.1%}, mean set size={np.mean(sizes):.2f}")
    # compare: singleton rate (definitive predictions)
    print(f"Singleton sets: {np.mean([s==1 for s in sizes]):.1%} of test epochs")
except NameError as e:
    print("Run section 18 first:",e)

## 18e · Deployment benchmarks

Model size, parameter count, ONNX export + ONNX Runtime latency per epoch —
the numbers that matter for edge deployment.

In [ ]:
import torch, numpy as np, time, os
ckpt=torch.load("models/fusion_model.pt",map_location="cpu",weights_only=False)
print("checkpoint size:",round(os.path.getsize("models/fusion_model.pt")/1e6,2),"MB")
for i,rec in enumerate(ckpt["folds"]):
    n_params=sum(v.numel() for v in rec["model_state"].values())
    print(f"fold {i+1}: {n_params/1e3:.0f}K params | seq_len={rec.get('seq_len',1)} | ablate={rec.get('ablate')}")
# ONNX Runtime latency (if onnx exported in section 16)
try:
    import onnxruntime as ort
    sess=ort.InferenceSession("exports/fusion_fold0.onnx")
    xe=np.random.randn(1,len(ckpt["eeg_cols"])).astype(np.float32)
    ye=np.random.randn(1,len(ckpt["eye_cols"])).astype(np.float32)
    sess.run(None,{"eeg":xe,"eye":ye})  # warmup
    t0=time.time()
    for _ in range(200): sess.run(None,{"eeg":xe,"eye":ye})
    print(f"ONNX inference: {(time.time()-t0)/200*1000:.2f} ms/epoch")
except Exception as e:
    print("ONNX benchmark skipped:",type(e).__name__)

## 18f · Domain-adversarial training (GRL)

`--grl` adds a subject discriminator on the fused embedding with a gradient
reversal layer — the encoder learns features that predict load but *not* which
subject produced them. Targets the biggest real-data problem: subject shift.
Mixup is auto-disabled (conflicts with subject labels). Tune `--grl-lambda`
(reversal strength) and `--grl-weight` (loss weight).

In [ ]:
# Domain-adversarial run — compare vs baseline on the SAME holdout
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 5 --grl --grl-lambda 1.0 --grl-weight 0.1                     --out models/fusion_grl
import torch
for name in ["fusion_model","fusion_grl"]:
    try:
        c=torch.load(f"models/{name}.pt",map_location="cpu",weights_only=False)
        s=c["cv_summary"]; print(f"{name:15s} acc={s['mean_accuracy']:.3f} f1={s['mean_macro_f1']:.3f} std={s['std_accuracy']:.3f}")
    except Exception as e: print(name,"->",e)

## 18g · Weight averaging (SWA)

`--swa` averages weights across eval snapshots and keeps the average only if it
beats the best single checkpoint on validation F1 — free accuracy when the loss
landscape is flat. Combine freely with `--grl`, `--seq-len`, `--use-gbm`.

In [ ]:
# SWA run — same holdout, compare vs baseline
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 5 --swa --use-gbm --out models/fusion_swa
import torch
for name in ["fusion_model","fusion_swa"]:
    try:
        c=torch.load(f"models/{name}.pt",map_location="cpu",weights_only=False)
        s=c["cv_summary"]; print(f"{name:15s} acc={s['mean_accuracy']:.3f} f1={s['mean_macro_f1']:.3f} std={s['std_accuracy']:.3f}")
    except Exception as e: print(name,"->",e)

## 18h · Test-time augmentation & per-class calibration

**TTA** — average predictions over feature-noise perturbations; usually a
small free accuracy gain. **Per-class temperature** — separate calibration
per class fixes `medium`-specific over/under-confidence.

In [ ]:
try:
    # --- TTA: average probs over gaussian feature jitter ---
    import torch, numpy as np
    from sklearn.metrics import accuracy_score
    fold=bundle["folds"][0]; _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    Xt=torch.tensor(xe,dtype=torch.float32); Yt=torch.tensor(ye,dtype=torch.float32)
    yt=te["label"].values; rng=np.random.default_rng(0)
    with torch.no_grad():
        base=torch.softmax(fold["temp"](fold["model"](Xt,Yt)),1).numpy()
        tta=np.zeros_like(base)
        for _ in range(10):
            lo=fold["temp"](fold["model"](Xt+rng.normal(0,0.05,Xt.shape).astype(np.float32)*torch.tensor(Xt.std(0)),
                                        Yt+rng.normal(0,0.05,Yt.shape).astype(np.float32)*torch.tensor(Yt.std(0))))
            tta+=torch.softmax(lo,1).numpy()
        tta/=10
    p0=np.array(classes)[base.argmax(1)]; p1=np.array(classes)[tta.argmax(1)]
    print(f"TTA: base acc={accuracy_score(yt,p0):.3f} -> tta acc={accuracy_score(yt,p1):.3f}")
except NameError as e:
    print("Run section 9 first:",e)

In [ ]:
try:
    # --- per-class temperature scaling on pooled preds ---
    import numpy as np, torch, torch.nn as nn
    y_idx=np.array([classes.index(t) for t in y_arr]); P=P_arr; conf=P.max(1)
    # 1-param temperature per class, fit on calibration half
    rng=np.random.default_rng(0); n=len(y_idx); idx=rng.permutation(n)
    cal,tst=idx[:n//2],idx[n//2:]
    T={}
    for c in range(len(classes)):
        t=nn.Parameter(torch.tensor(1.5))
        opt=torch.optim.LBFGS([t],lr=0.05,max_iter=100)
        logits=torch.log(torch.tensor(P[cal])+1e-9)
        def closure():
            opt.zero_grad(); loss=nn.functional.cross_entropy(logits/t.clamp(min=.5),torch.tensor(y_idx[cal])); loss.backward(); return loss
        opt.step(closure); T[classes[c]]=float(t.clamp(min=.5))
    # apply per-sample T of predicted class
    Pt=P_arr.copy()
    for c,cname in enumerate(classes):
        m=(P_arr.argmax(1)==c)
        Pt[m]=torch.softmax(torch.tensor(np.log(P_arr[m]+1e-9)/T[cname]),1).numpy()
    acc0=accuracy_score(y_arr,np.array(classes)[P_arr.argmax(1)])
    acc1=accuracy_score(y_arr,np.array(classes)[Pt.argmax(1)])
    print(f"per-class temp: acc {acc0:.3f} -> {acc1:.3f} | temps={ {k:round(v,2) for k,v in T.items()} }")
except NameError as e:
    print("Run section 18 first:",e)

## 18i · Model pruning

Magnitude pruning: zero the smallest weights, measure accuracy vs sparsity.
Shows how compressible the fused model is before fine-tuning or quantization.

In [ ]:
try:
    import torch, torch.nn as nn, numpy as np, copy
    from sklearn.metrics import accuracy_score
    fold=bundle["folds"][0]; _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    Xt=torch.tensor(xe,dtype=torch.float32); Yt=torch.tensor(ye,dtype=torch.float32); yt=te["label"].values
    m=fold["model"]
    # collect 2D+ weight tensors
    keys=[k for k,v in m.state_dict().items() if v.dim()>=2]
    allw=torch.cat([m.state_dict()[k].abs().flatten() for k in keys])
    def acc_with(thresh):
        m2=copy.deepcopy(m); m2.eval(); sd=m2.state_dict()
        with torch.no_grad():
            for k in keys: sd[k].mul_((sd[k].abs()>=thresh).float())
        with torch.no_grad():
            lo=fold["temp"](m2(Xt,Yt)); pr=np.array(classes)[torch.softmax(lo,1).numpy().argmax(1)]
        return accuracy_score(yt,pr)
    base=acc_with(0.0)
    for pct in (50,70,90):
        t=torch.quantile(allw,pct/100).item()
        print(f"prune {pct}% (|w|>={t:.4f}): acc={acc_with(t):.3f}  (base {base:.3f})")
    print("Note: unstructured magnitude pruning without fine-tune — expect graceful degradation")
except NameError as e:
    print("Run section 9 first:",e)

## 18j · Knowledge distillation

`--teacher <ckpt>` trains a student against the teacher's temperature-softened
probabilities (`--kd-alpha` mixes hard+soft loss, `--kd-temp` controls softness).
Use a big/ensemble model to distill a small deployable student — or distill the
fused model into an EEG-only student.

In [ ]:
# Distill the fused ensemble into a fresh student (same arch, fewer epochs needed)
!python src/train.py --features data/processed/features.parquet                     --epochs 100 --cv 5 --teacher models/fusion_model.pt                     --kd-alpha 0.7 --kd-temp 2.0 --out models/fusion_student
import torch
for name in ["fusion_model","fusion_student"]:
    try:
        c=torch.load(f"models/{name}.pt",map_location="cpu",weights_only=False)
        s=c["cv_summary"]; print(f"{name:15s} acc={s['mean_accuracy']:.3f} f1={s['mean_macro_f1']:.3f}")
    except Exception as e: print(name,"->",e)

## 18k · Quantization (INT8)

Dynamic post-training quantization of the Linear layers — measures the
size/accuracy/latency tradeoff for edge deployment.

In [ ]:
try:
    import torch, torch.nn as nn, numpy as np, io, time, warnings
    warnings.filterwarnings("ignore")
    import torch.quantization as quant
    fold=bundle["folds"][0]; _,te=splits[0]; te=te.copy(); te["norm_key"]=make_norm_key(te)
    eeg_n,_=adaptive_subject_norm(coerce_numeric(te,eeg_cols),te["norm_key"])
    eye_n,_=adaptive_subject_norm(coerce_numeric(te,eye_cols),te["norm_key"])
    xe=np.nan_to_num(fold["scaler_eeg"].transform(eeg_n)); ye=np.nan_to_num(fold["scaler_eye"].transform(eye_n))
    Xt=torch.tensor(xe,dtype=torch.float32); Yt=torch.tensor(ye,dtype=torch.float32); yt=te["label"].values
    m=fold["model"]; m.eval()
    from sklearn.metrics import accuracy_score
    def acc_of(model):
        with torch.no_grad():
            lo=fold["temp"](model(Xt,Yt)); return accuracy_score(yt,np.array(classes)[torch.softmax(lo,1).numpy().argmax(1)])
    acc_fp=acc_of(m)
    mq=quant.quantize_dynamic(m,{nn.Linear},dtype=torch.qint8)
    acc_q=acc_of(mq)
    buf=io.BytesIO(); torch.save(mq.state_dict(),buf)
    # latency
    with torch.no_grad():
        m(Xt,Yt); mq(Xt,Yt)
        t0=time.time()
        for _ in range(100): m(Xt,Yt)
        t_fp=(time.time()-t0)/100*1000
        t0=time.time()
        for _ in range(100): mq(Xt,Yt)
        t_q=(time.time()-t0)/100*1000
    print(f"FP32 : acc={acc_fp:.3f} | {t_fp:.2f} ms/epoch")
    print(f"INT8 : acc={acc_q:.3f} | {t_q:.2f} ms/epoch | state_dict {len(buf.getvalue())/1e6:.2f} MB")
except NameError as e:
    print("Run section 9 first:",e)